# Trabajo en clase — Q-Learning con FrozenLake

En CheeseWorld construimos el algoritmo desde cero. Ahora utilizaremos el mismo procedimiento en un ambiente estándar de **Gymnasium**.

## Objetivo

Durante la clase debes relacionar cada parte del código con los conceptos:

- estado \(s\);
- acción \(a\);
- recompensa \(r\);
- Q-table;
- exploración y explotación;
- TD target;
- TD error;
- política greedy.

Este notebook tiene **espacios para discutir y escribir conclusiones durante la clase**.


## 1. Imports y funciones de Q-Learning


In [1]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output, display

from IPython.display import HTML
from matplotlib import animation
import matplotlib.pyplot as plt


def play_episode(env, Q=None, random_policy=False, max_steps=100, seed=None):
    """Ejecuta un episodio y devuelve sus frames y recompensa total."""
    state, _ = env.reset(seed=seed)
    frames = [env.render()]
    total_reward = 0.0

    for _ in range(max_steps):
        if random_policy:
            action = env.action_space.sample()
        else:
            q_values = Q[state]
            max_q = np.max(q_values)

            # Desempate aleatorio entre acciones con el mismo Q.
            best_actions = np.flatnonzero(q_values == max_q)
            action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())
        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=700):
    """Convierte una lista de frames RGB en una animación reproducible en Jupyter."""
    fig = plt.figure(figsize=(4, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state):
    # Si hay empate entre varias acciones con el mismo Q, desempata al azar.
    max_q = np.max(Qtable[state])
    best_actions = np.flatnonzero(Qtable[state] == max_q)
    return int(np.random.choice(best_actions))


def epsilon_greedy_policy(Qtable, state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()

    return greedy_policy(Qtable, state)


def train_q_learning(
    env,
    Qtable,
    n_episodes=5000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    max_steps=100,
    start_episode=0,
):
    rewards = []

    for episode in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        global_episode = start_episode + episode
        epsilon = min_epsilon + (
            max_epsilon - min_epsilon
        ) * np.exp(-decay_rate * global_episode)

        for _ in range(max_steps):
            action = epsilon_greedy_policy(
                Qtable, state, epsilon, env
            )

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            best_next_q = 0.0 if done else np.max(Qtable[next_state])

            td_target = reward + gamma * best_next_q
            td_error = td_target - Qtable[state, action]

            Qtable[state, action] += learning_rate * td_error

            state = next_state
            total_reward += reward

            if done:
                break

        rewards.append(total_reward)

    return Qtable, rewards


def evaluate_q_policy(env, Qtable, n_episodes=100, max_steps=100):
    episode_rewards = []

    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = greedy_policy(Qtable, state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            state = next_state
            total_reward += reward

            if terminated or truncated:
                break

        episode_rewards.append(total_reward)

    return np.mean(episode_rewards), np.std(episode_rewards)


def show_frame(env, title=''):
    frame = env.render()
    plt.figure(figsize=(4, 4))
    plt.imshow(frame)
    plt.axis('off')
    plt.title(title)
    display(plt.gcf())
    plt.close()

In [2]:
# Semillas para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### 💬 Antes de ejecutar

En CheeseWorld teníamos explícitamente una clase `Environment` y una clase `QLearningAgent`.

**Pregunta:** en este notebook, ¿qué papel cumple Gymnasium y dónde quedó representado el agente?

**Notas:**

1. **Gymnasium hace el papel de la clase `Environment`.** Guarda el estado del mundo, define el espacio de estados y de acciones, y con `env.step(action)` aplica la dinámica: devuelve el siguiente estado, la recompensa y si el episodio terminó. `env.reset()` reinicia el episodio y `env.render()` dibuja el mapa. Nosotros ya no programamos las reglas del lago, solo interactuamos con ellas.
2. **El agente quedó repartido entre la Q-table y las funciones.** Ya no hay una clase `QLearningAgent`: su "memoria" es el arreglo `Qtable` (una matriz de NumPy) y su comportamiento son las funciones `greedy_policy` y `epsilon_greedy_policy` (cómo decide) y `train_q_learning` (cómo aprende con la regla TD).
3. La separación conceptual se mantiene igual que en CheeseWorld: el ambiente no sabe nada de Q, y el agente no conoce las probabilidades de transición; solo ve las tuplas $(s,a,r,s')$ que le devuelve `step`.


## 2. Crear FrozenLake


In [3]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False,
    render_mode="rgb_array"
)

state, info = env.reset(seed=SEED)
env.action_space.seed(SEED)

print("Estado inicial:", state)
print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)

Estado inicial: 0
Número de estados: 16
Número de acciones: 4


En FrozenLake las acciones son:

| Acción | Código |
|---|---:|
| Left | 0 |
| Down | 1 |
| Right | 2 |
| Up | 3 |

Primero trabajaremos con `is_slippery=False`, es decir, con transiciones determinísticas.


### 💬 Actividad 1 — La Q-table

Antes de crearla:

1. ¿Cuántas filas debe tener la Q-table?
2. ¿Cuántas columnas?
3. ¿Qué representa una celda \(Q[s,a]\)?

**Respuesta / discusión:**

1. **16 filas**, una por estado. El mapa 4x4 tiene 16 casillas y el estado es simplemente el índice de la casilla, $s = 4\cdot\text{fila} + \text{columna}$.
2. **4 columnas**, una por acción: Left (0), Down (1), Right (2), Up (3). Por eso la tabla tiene forma `(16, 4)`.
3. $Q[s,a]$ es la estimación del **retorno descontado esperado** si el agente está en la casilla $s$, ejecuta la acción $a$ y luego sigue la mejor política: $Q(s,a) \approx \mathbb{E}[r_{t+1} + \gamma r_{t+2} + \gamma^2 r_{t+3} + \cdots \mid s_t=s,\ a_t=a]$. Como en FrozenLake la única recompensa es $+1$ al llegar a la meta, en el caso determinístico $Q(s,a) = \gamma^{k}$, donde $k+1$ es el número de pasos que faltan para llegar a la meta.


In [4]:
state_space = env.observation_space.n
action_space = env.action_space.n

Q = initialize_q_table(state_space, action_space)

print("Q-table shape:", Q.shape)
Q

Q-table shape: (16, 4)


array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

### 💬 Actividad 2 — Inicio del aprendizaje

Todos los valores son cero.

$$
Q(s,a)=0
$$

¿Esto significa que todas las acciones son malas, o que el agente todavía no sabe nada?

¿Qué ocurre si varias acciones tienen exactamente el mismo valor máximo?

**Notas:**

1. **Significa que el agente todavía no sabe nada**, no que las acciones sean malas. El cero es solo el valor inicial que escogimos; no proviene de ninguna experiencia. De hecho, en FrozenLake el cero coincide con el valor de caer en un hueco, así que al principio el agente no puede distinguir entre "esta acción lleva a un hueco" y "nunca he probado esta acción".
2. **Si varias acciones empatan en el máximo**, `argmax` de NumPy siempre devolvería la primera (Left), y el agente quedaría atrapado repitiendo la misma acción contra la pared. Por eso `greedy_policy` hace un **desempate aleatorio** con `np.flatnonzero(Q[s] == max_q)`. Al inicio todas empatan, así que la política greedy se comporta como una política aleatoria.
3. Esto también explica por qué hace falta explorar: mientras la recompensa de $+1$ no se haya encontrado al menos una vez, toda la tabla sigue en cero y no hay ninguna señal que seguir.


## 3. Ejecutar una transición


In [5]:
state, _ = env.reset()

epsilon = 1.0
action = epsilon_greedy_policy(Q, state, epsilon, env)

next_state, reward, terminated, truncated, _ = env.step(action)

print("state      =", state)
print("action     =", action)
print("reward     =", reward)
print("next_state =", next_state)
print("done       =", terminated or truncated)

state      = 0
action     = 0
reward     = 0
next_state = 0
done       = False


### 💬 Actividad 3 — Identificar la experiencia

Escribe la experiencia anterior como:

$$
(s,a,r,s')
$$

**Experiencia:**

$$
(s,a,r,s') = (0,\ 0,\ 0,\ 0)
$$

El agente estaba en la casilla inicial (0), escogió Left (0), chocó contra el borde, no recibió recompensa (0) y quedó en la misma casilla (0). Con $\epsilon = 1.0$ la acción fue completamente aleatoria.

¿De cuál de esos cuatro elementos **no disponíamos directamente** en Value Iteration cuando hablábamos de experiencia real?


**Respuesta:**

El elemento que **no teníamos directamente** es $s'$, el siguiente estado observado. En Value Iteration no se interactúa con el ambiente: se usa el modelo completo $P(s' \mid s,a)$ y $R(s,a,s')$ y se calcula la esperanza sobre **todos** los posibles $s'$:

$$V(s) \leftarrow \max_a \sum_{s'} P(s'\mid s,a)\left[R(s,a,s') + \gamma V(s')\right]$$

En Q Learning no conocemos $P$; en cambio el ambiente nos entrega **una muestra** concreta de $s'$ (y de $r$). Esa muestra reemplaza a la suma ponderada por probabilidades, y al promediar muchas actualizaciones con tasa $\alpha$ se aproxima la misma esperanza.

## 4. Del azar a una política aprendida

Vamos a observar **el mismo agente en tres momentos**. Primero no sabe nada y actúa al azar; luego veremos su política después de pocas experiencias; finalmente veremos la política después del entrenamiento completo.


In [6]:
# Guardaremos tres momentos del aprendizaje

# Momento 1: sin entrenamiento
Q_initial = initialize_q_table(
    env.observation_space.n,
    env.action_space.n
)

# Momento 2: poco entrenamiento
EARLY_EPISODES = 50
Q_early = Q_initial.copy()
Q_early, rewards_early = train_q_learning(
    env,
    Q_early,
    n_episodes=EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

# Momento 3: continuar hasta 10 000 episodios
TOTAL_EPISODES = 10000
Q_trained = Q_early.copy()
Q_trained, rewards_final = train_q_learning(
    env,
    Q_trained,
    n_episodes=TOTAL_EPISODES - EARLY_EPISODES,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
    start_episode=EARLY_EPISODES,
)

Q = Q_trained
rewards = rewards_early + rewards_final

print('Snapshots guardados:')
print('Q_initial : 0 episodios')
print(f'Q_early   : {EARLY_EPISODES} episodios')
print(f'Q_trained : {TOTAL_EPISODES} episodios')


Snapshots guardados:
Q_initial : 0 episodios
Q_early   : 50 episodios
Q_trained : 10000 episodios


### Momento 1 — Sin entrenamiento: random walk
Todavía no usamos la Q-table para decidir. Cada acción se selecciona aleatoriamente. Observa cómo interactúa el agente con el mundo.

In [7]:
frames_random, reward_random = play_episode(
    env,
    Q_initial,
    random_policy=True,
    seed=7
)

print(f"Recompensa total: {reward_random}")
frames_to_video(frames_random, interval=700)


Recompensa total: 0.0


### Momento 2 — Después de pocas iteraciones

Ahora el agente usa de forma **greedy** lo que ha aprendido en `Q_early`. Todavía conoce poco del ambiente, así que su comportamiento puede ser incompleto o equivocarse.


In [8]:
frames_early, reward_early = play_episode(
    env,
    Q_early,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_early}")
frames_to_video(frames_early, interval=700)


Recompensa total: 1.0

### Momento 3 — Agente entrenado

Finalmente usamos `Q_trained`. Ya no exploramos: en cada estado el agente selecciona una de las acciones con mayor valor $Q(s,a)$.


In [9]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    random_policy=False,
    seed=7
)

print(f"Recompensa total: {reward_trained}")
frames_to_video(frames_trained, interval=700)


Recompensa total: 1.0


### 💬 Actividad — ¿Qué cambió?

Compara las tres ejecuciones. El ambiente, los estados y las acciones son los mismos. **¿Qué cambió internamente en el agente para que su comportamiento mejore?**

Observa `Q_initial`, `Q_early` y `Q_trained` y relaciona sus valores con las acciones que viste ejecutar.


In [10]:
arrows = np.array(["←", "↓", "→", "↑"])
desc = env.unwrapped.desc.astype(str)

def show_policy(Qtable, name):
    print(f"Política greedy de {name} (· = empate total en ceros):")
    for r in range(4):
        row = []
        for c in range(4):
            s = r * 4 + c
            if desc[r, c] in "HG":
                row.append(desc[r, c])
            elif np.all(Qtable[s] == Qtable[s].max()):
                row.append("·")
            else:
                row.append(arrows[np.argmax(Qtable[s])])
        print("   " + " ".join(row))
    print(f"   estados con algún Q distinto de 0: {np.sum(np.any(Qtable != 0, axis=1))} de {Qtable.shape[0]}\n")

for Qx, name in [(Q_initial, "Q_initial"), (Q_early, "Q_early"), (Q_trained, "Q_trained")]:
    show_policy(Qx, name)

np.set_printoptions(precision=3, suppress=True)
print("Q_early:\n", Q_early)
print("\nQ_trained:\n", Q_trained)

Política greedy de Q_initial (· = empate total en ceros):
   · · · ·
   · H · H
   · · · H
   H · · G
   estados con algún Q distinto de 0: 0 de 16

Política greedy de Q_early (· = empate total en ceros):
   · · · ·
   · H · H
   · · · H
   H · → G
   estados con algún Q distinto de 0: 1 de 16

Política greedy de Q_trained (· = empate total en ceros):
   ↓ → ↓ ←
   ↓ H ↓ H
   → ↓ ↓ H
   H → → G
   estados con algún Q distinto de 0: 11 de 16

Q_early:
 [[0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.  0.  0.7 0. ]
 [0.  0.  0.  0. ]]

Q_trained:
 [[0.735 0.774 0.774 0.735]
 [0.735 0.    0.815 0.774]
 [0.774 0.857 0.774 0.815]
 [0.815 0.    0.774 0.774]
 [0.774 0.815 0.    0.735]
 [0.    0.    0.    0.   ]
 [0.    0.902 0.    0.815]
 [0.    0.    0.    0.   ]
 [0.815 0.

**Respuesta:**

Lo único que cambió fue **la Q-table**; el ambiente, la función de decisión greedy y las acciones disponibles son exactamente los mismos.

1. **`Q_initial`** tiene las 64 entradas en cero. Todas las acciones empatan, así que el agente actúa al azar. En el video del momento 1 (política aleatoria) no llega a la meta y la recompensa es 0.
2. **`Q_early`** (50 episodios) tiene **una sola entrada distinta de cero**: $Q(14,\text{Right}) = 0.7$, que es $\alpha \cdot 1$ después de haber llegado una vez a la meta desde la casilla 14. En los otros 15 estados sigue habiendo empate, así que fuera de esa casilla el agente todavía camina al azar. En nuestra ejecución el episodio con `seed=7` sí llegó a la meta, pero fue **suerte del desempate aleatorio**, no conocimiento: en 50 episodios $\epsilon$ seguía cerca de $0.95$ y el agente casi solo exploró.
3. **`Q_trained`** (10 000 episodios) tiene valores en los 11 estados no terminales, y siguen el patrón $\gamma^k$: $1.0$ junto a la meta, $0.95$, $0.9025$, … hasta $0.774 = 0.95^5$ en la casilla inicial, lo que corresponde al camino más corto de 6 pasos. Las acciones que llevan a un hueco quedaron en 0 y las que se alejan de la meta quedan un factor $\gamma$ por debajo. La política greedy resultante es un camino directo que rodea los huecos, y eso es exactamente lo que se ve en el video del momento 3.

En resumen: la experiencia fue propagando el $+1$ de la meta hacia atrás, casilla por casilla, a través del término $\gamma \max_{a'} Q(s',a')$, y la política greedy simplemente lee esos números.

### 💬 Actividad 4 — Leer una fila de Q

Selecciona un estado $s$ y observa:

$$
Q(s,0), Q(s,1), Q(s,2), Q(s,3)
$$

**Estado seleccionado:** $s = 14$ (fila 3, columna 2, justo a la izquierda de la meta)

**Valores Q:**

| Acción | $Q(14,a)$ |
|---|---:|
| Left | 0.9025 |
| Down | 0.9500 |
| Right | **1.0000** |
| Up | 0.9025 |

¿Cuál acción seleccionaría $\arg\max_a Q(s,a)$? **Right**.

**Interpretación:**

> Right lleva directamente a la meta y recibe $r=1$, y como el episodio termina no hay valor futuro: $Q = 1$. Down choca con el borde inferior y el agente se queda en 14, así que vale $\gamma \cdot 1 = 0.95$ (pierde un paso). Left y Up lo alejan una casilla, por lo que tardaría dos pasos más: $\gamma^2 = 0.9025$. Los valores no indican qué tan "buena" es la acción en el momento, sino cuántos pasos de retraso introduce en el camino hacia la meta. En el estado inicial (0), Down y Right empatan en $0.774 = \gamma^5$ porque ambas inician un camino óptimo de 6 pasos.


In [11]:
s = 14   # casilla justo a la izquierda de la meta (fila 3, columna 2)
for name, q in zip(["Left", "Down", "Right", "Up"], Q_trained[s]):
    print(f"{name:5s}: {q:.4f}")
print("argmax:", ["Left", "Down", "Right", "Up"][int(np.argmax(Q_trained[s]))])

print()
s0 = 0   # estado inicial
for name, q in zip(["Left", "Down", "Right", "Up"], Q_trained[s0]):
    print(f"{name:5s}: {q:.4f}")

Left : 0.9025
Down : 0.9500
Right: 1.0000
Up   : 0.9025
argmax: Right

Left : 0.7351
Down : 0.7738
Right: 0.7738
Up   : 0.7351


## 5. Evaluar la política aprendida


In [12]:
mean_reward, std_reward = evaluate_q_policy(
    env,
    Q_trained,
    n_episodes=100
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")


Mean reward: 1.000
Std reward : 0.000


### 💬 Actividad 5 — Exploration vs. exploitation

Durante entrenamiento usamos $\epsilon$-greedy.

Durante evaluación usamos:

$$
a=\arg\max_aQ(s,a)
$$

¿Por qué **no exploramos** durante la evaluación?

**Conclusión:**

> La exploración sirve para **aprender**: obliga al agente a probar acciones cuyo valor todavía no conoce para no quedarse con una estimación pobre. En la evaluación ya no queremos aprender, queremos **medir** la calidad de la política que se obtuvo, y la política aprendida es la greedy, $\pi(s) = \arg\max_a Q(s,a)$. Si siguiéramos explorando, una fracción $\epsilon$ de las acciones sería aleatoria y la medida mezclaría la calidad de la política con el ruido de la exploración; en FrozenLake una acción aleatoria puede tirar al agente a un hueco y subestimaríamos el desempeño. Además, durante la evaluación no se actualiza Q, así que explorar no aportaría ninguna información útil. Con política greedy obtuvimos recompensa media de 1.000 y desviación 0.000: el agente llega a la meta en los 100 episodios.


## 6. Experimento: FrozenLake estocástico


In [13]:
slippery_env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=True,
    render_mode="rgb_array"
)
slippery_env.reset(seed=SEED)
slippery_env.action_space.seed(SEED)

Q_slippery = initialize_q_table(
    slippery_env.observation_space.n,
    slippery_env.action_space.n
)

Q_slippery, rewards_slippery = train_q_learning(
    slippery_env,
    Q_slippery,
    n_episodes=20000,
    learning_rate=0.7,
    gamma=0.95,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.0005,
    max_steps=100
)

mean_reward, std_reward = evaluate_q_policy(
    slippery_env,
    Q_slippery,
    n_episodes=500
)

print(f"Mean reward: {mean_reward:.3f}")
print(f"Std reward : {std_reward:.3f}")

Mean reward: 0.702
Std reward : 0.457


In [14]:
show_policy(Q_slippery, "Q_slippery")
print("Política del caso determinístico, para comparar:")
show_policy(Q_trained, "Q_trained")

Política greedy de Q_slippery (· = empate total en ceros):
   ← ↑ ← ↑
   ← H → H
   ↑ ↓ ← H
   H → ↓ G
   estados con algún Q distinto de 0: 11 de 16

Política del caso determinístico, para comparar:
Política greedy de Q_trained (· = empate total en ceros):
   ↓ → ↓ ←
   ↓ H ↓ H
   → ↓ ↓ H
   H → → G
   estados con algún Q distinto de 0: 11 de 16



### Experimento adicional: variabilidad con el hielo resbaloso

Al repetir el entrenamiento resbaloso con otras semillas el resultado cambia muchísimo. Para entender por qué, entrenamos con 5 semillas usando el $\alpha = 0.7$ original y un $\alpha = 0.1$ más pequeño (todo lo demás igual). Esta celda tarda un par de minutos.

In [15]:
def run_slippery(lr, seed):
    random.seed(seed); np.random.seed(seed)
    e = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
    e.reset(seed=seed); e.action_space.seed(seed)
    Qs, _ = train_q_learning(e, initialize_q_table(16, 4), n_episodes=20000,
                             learning_rate=lr, gamma=0.95, max_epsilon=1.0,
                             min_epsilon=0.05, decay_rate=0.0005, max_steps=100)
    return evaluate_q_policy(e, Qs, n_episodes=500)[0]

for lr in [0.7, 0.1]:
    results = [run_slippery(lr, seed) for seed in range(5)]
    print(f"alpha = {lr}: éxito por semilla = {np.round(results, 3)}, "
          f"media = {np.mean(results):.3f}, desv. = {np.std(results):.3f}")

alpha = 0.7: éxito por semilla = [0.158 0.69  0.352 0.646 0.748], media = 0.519, desv. = 0.226


alpha = 0.1: éxito por semilla = [0.748 0.63  0.774 0.754 0.756], media = 0.732, desv. = 0.052


### 💬 Actividad 6 — Determinístico vs. estocástico

Compara:

- `is_slippery=False`
- `is_slippery=True`

¿Qué cambia en el **ambiente**?

¿Qué cambia en la **ecuación de Q-Learning**?

**Discusión:**

**Ambiente.** Con `is_slippery=False` las transiciones son determinísticas: la acción escogida siempre se ejecuta. Con `is_slippery=True` el hielo resbala: la acción deseada se ejecuta con probabilidad $1/3$ y con probabilidad $1/3$ cada una el agente se mueve en una de las dos direcciones perpendiculares. Ahora $P(s' \mid s,a)$ es una distribución y la misma acción en la misma casilla puede llevar a sitios distintos, incluso a un hueco. Por eso el resultado empeora mucho: la recompensa media pasó de **1.000** a **0.702** (desviación 0.457), es decir, el agente llega a la meta en cerca del 70 % de los episodios. No es un fallo del entrenamiento: ni siquiera la política óptima garantiza llegar siempre.

La política aprendida también cambia de forma. En el caso resbaloso aparecen acciones que parecen absurdas, como Left en la casilla inicial o Up en la fila superior. Son estrategias conservadoras: al empujar contra una pared, los resbalones perpendiculares nunca llevan hacia el hueco, así que el agente prefiere avanzar despacio pero seguro.

**Algoritmo.** La ecuación de Q Learning **no cambia en nada**:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\left[r + \gamma \max_{a'}Q(s',a') - Q(s,a)\right]$$

Lo que cambia es su interpretación: como $s'$ ahora es aleatorio, el target TD es una muestra ruidosa y la actualización con tasa $\alpha$ actúa como un promedio móvil que aproxima la esperanza $\mathbb{E}_{s'}[r + \gamma \max_{a'}Q(s',a')]$. Ese es justamente el sentido de que el método sea model free: la estocasticidad se absorbe a través de las muestras sin conocer $P$. En la práctica sí cambian los hiperparámetros: entrenamos 20 000 episodios con un decaimiento de $\epsilon$ más lento (0.0005) porque se necesitan muchas más muestras. Además, $\alpha$ pasa a ser crítico. Con $\alpha = 0.7$ cada actualización le da mucho peso a la última muestra, y una sola caída por resbalón borra casi todo lo aprendido sobre esa acción. El experimento adicional lo confirma: con $\alpha = 0.7$ la tasa de éxito varió entre 0.158 y 0.748 según la semilla (media 0.519, desviación 0.226), mientras que con $\alpha = 0.1$ quedó entre 0.630 y 0.774 (media 0.732, desviación 0.052). En el caso determinístico un $\alpha$ grande no tiene costo porque el target de una transición siempre es el mismo; en el estocástico conviene un $\alpha$ pequeño o decreciente para que la actualización realmente promedie.


# Cierre de clase

Completa antes de terminar:

**1. ¿Qué almacena $Q(s,a)$?**

> La estimación del retorno descontado esperado de tomar la acción $a$ en el estado $s$ y comportarse de manera óptima desde ahí. No es la recompensa inmediata, sino el valor a largo plazo de esa decisión.

**2. ¿De dónde sale $\max_{a'}Q(s',a')$?**

> De la propia Q-table, evaluada en el estado $s'$ que el ambiente acaba de devolver. Es la estimación actual del valor del mejor paso siguiente (bootstrapping): usamos una estimación para mejorar otra. Viene de la ecuación de optimalidad de Bellman, $Q^*(s,a) = \mathbb{E}[r + \gamma \max_{a'} Q^*(s',a')]$, y el máximo es lo que hace que Q Learning sea off policy: aprende el valor de la política greedy aunque se comporte con $\epsilon$ greedy. Si $s'$ es terminal se toma como 0.

**3. ¿Por qué necesitamos $\epsilon$-greedy?**

> Para equilibrar exploración y explotación. Si el agente fuera siempre greedy, repetiría las primeras acciones que le dieron algo y nunca descubriría caminos mejores (o, con la tabla en ceros, nunca encontraría la meta de forma dirigida). Con probabilidad $\epsilon$ prueba una acción al azar y con $1-\epsilon$ usa lo aprendido. Haciendo que $\epsilon$ decaiga, el agente explora mucho al principio, cuando no sabe nada, y explota cada vez más a medida que sus estimaciones mejoran.

**4. ¿Por qué Q-Learning es model-free?**

> Porque nunca usa ni estima las probabilidades de transición $P(s'\mid s,a)$ ni la función de recompensa $R$. Aprende directamente los valores $Q(s,a)$ a partir de experiencias $(s,a,r,s')$ obtenidas interactuando con el ambiente. La prueba está en este mismo notebook: el mismo código funcionó sin modificaciones en el lago determinístico y en el resbaloso, aunque sus dinámicas son distintas. En Value Iteration, en cambio, necesitábamos conocer el modelo completo para calcular la esperanza sobre $s'$.
